In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['hatch.linewidth'] = 0.2
import numpy as np
import polars as pl

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.signal_categories import del1g_detailed_category_labels, del1g_detailed_category_colors, del1g_detailed_category_labels_latex
from src.signal_categories import del1g_detailed_category_hatches, del1g_detailed_categories_dic, del1g_detailed_category_queries
from src.signal_categories import train_category_labels, train_category_labels_latex

from src.file_locations import intermediate_files_location

from src.plot_helpers import make_histogram_plot

from src.ntuple_variables.variables import combined_training_vars

from src.df_helpers import lazy_height


In [ ]:
# currently not handling DetVar


In [ ]:
training = "all_vars"
training_vars = combined_training_vars
reco_categories = train_category_labels
reco_category_labels_latex = train_category_labels_latex

# File Loading

In [ ]:
# Load the normal df (lazy)
print("Loading all_df.parquet...")
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
print(f"Total events: {lazy_height(all_df)}")

# Load the BDT predictions df (lazy)
print("Loading predictions.parquet...")
pred_bdt_scores_df = pl.scan_parquet(f"../training_outputs/{training}/predictions.parquet")
print(f"Total events: {lazy_height(pred_bdt_scores_df)}")

# Load the spline df (lazy)
print("Loading spline_weights_df.parquet...")
spline_weights_df = pl.scan_parquet(f"{intermediate_files_location}/spline_weights_df.parquet")
print(f"Total events: {lazy_height(spline_weights_df)}")


In [ ]:
print("merging all_df and predictions.pkl...")
merged_df_no_data_drop = all_df.join(
    pred_bdt_scores_df, 
    on=["filetype", "run", "subrun", "event"], 
    how="left"
)

print(f"merged_df_no_data_drop total in-FV NC Coherent 1g reweighted events: {merged_df_no_data_drop.filter((pl.col('filetype') == 'NC_coherent_1g_reweighted') & pl.col('wc_truth_inFV')).collect()['wc_net_weight'].sum()}")
print(f"merged_df_no_data_drop num unweighted in-FV NC Coherent 1g reweighted events: {merged_df_no_data_drop.filter((pl.col('filetype') == 'NC_coherent_1g_reweighted') & pl.col('wc_truth_inFV')).collect().height}")

del all_df
del pred_bdt_scores_df


In [ ]:
# removing data and training-only events that aren't part of the prediction
full_pred = merged_df_no_data_drop.filter(
    ~pl.col("filetype").is_in(["data", "isotropic_one_gamma_overlay", "delete_one_gamma_overlay"])
)
full_data = merged_df_no_data_drop.filter(pl.col("filetype") == "data")

# adding empty score columns
prob_categories = ["prob_" + cat for cat in reco_categories]
for prob in prob_categories:
    full_pred = full_pred.with_columns(pl.col(prob).fill_null(-1))
    full_data = full_data.with_columns(pl.col(prob).fill_null(-1))

generic_pred_df = full_pred.filter(pl.col("wc_kine_reco_Enu") > 0)
non_generic_pred_df = full_pred.filter(pl.col("wc_kine_reco_Enu") < 0)
del full_pred

num_train_events = lazy_height(generic_pred_df.filter(pl.col("used_for_training") == True))
num_test_events = lazy_height(generic_pred_df.filter(pl.col("used_for_testing") == True))

#num_train_events = generic_pred_df.filter(pl.col("used_for_training") == True).height
#num_test_events = generic_pred_df.filter(pl.col("used_for_testing") == True).height
print(f"num_train_events: {num_train_events}, num_test_events: {num_test_events}")
frac_test = num_test_events / (num_train_events + num_test_events)
print(f"weighting up preselected prediction events by the fraction of test/train events: {frac_test:.3f}")

# Modify weights using polars expressions
generic_pred_df = generic_pred_df.with_columns(
    pl.when(pl.col("used_for_testing"))
    .then(pl.col("wc_net_weight") / frac_test)
    .otherwise(pl.col("wc_net_weight"))
    .alias("wc_net_weight")
)

generic_test_pred_df = generic_pred_df.filter(pl.col("used_for_testing") == True)
del generic_pred_df

generic_data_df = full_data.filter(pl.col("wc_kine_reco_Enu") > 0)
del full_data

generic_test_pred_data_df = pl.concat([generic_test_pred_df, generic_data_df])
del generic_test_pred_df
del non_generic_pred_df


In [ ]:
# Select physics variables (drop training-only variables to save memory)
load_vars = [col for col in generic_test_pred_data_df.collect_schema().names() if col not in combined_training_vars]

generic_test_pred_data_df = generic_test_pred_data_df.select(load_vars).collect()


In [ ]:
generic_test_pred_data_df.select(["filetype", "detailed_run_period", "run", "subrun", "event", "wc_kine_reco_Enu"])


# Generic All Runs

In [ ]:
make_histogram_plot(
    pred_and_data_sel_df=generic_test_pred_data_df,
    weights_df=None, # streamed lazily at plot generation
    use_rw_systematics=True,
    bins=np.linspace(0, 2000, 21),
    var="wc_kine_reco_Enu",
    display_var=r"WC Reconstructed $E_\nu$ (GeV)",
    title="Run 4a+4b WC Generic Neutrino Selection",
    selname="generic_presel"
)


# Generic Run 4b

In [ ]:
run4b_generic_test_pred_data_df = generic_test_pred_data_df.filter(pl.col("detailed_run_period")=="4b")

generic_test_pred_data_df.select(["filetype", "detailed_run_period", "run", "subrun", "event", "wc_kine_reco_Enu"])


In [ ]:
# problem, using non-4b runs for net_weight POT normalization!

make_histogram_plot(
    pred_and_data_sel_df=run4b_generic_test_pred_data_df,
    weights_df=None, # streamed lazily at plot generation
    use_rw_systematics=True,
    bins=np.linspace(0, 2000, 21),
    var="wc_kine_reco_Enu",
    display_var=r"WC Reconstructed $E_\nu$ (GeV)",
    title="Run 4b WC Generic Neutrino Selection",
    selname="generic_presel",
    net_weight_var="run4b_only_wc_net_weight"
)
